In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [2]:
pca_train = pd.read_csv("../data/processed/pca_train.csv")
pca_test = pd.read_csv("../data/processed/pca_test.csv")

print("PCA training shape:", pca_train.shape)
print("PCA testing shape:", pca_test.shape)

PCA training shape: (4000, 18)
PCA testing shape: (1000, 18)


In [3]:
X_train_pca = pca_train.drop("Adherence", axis=1)
y_train_pca = pca_train["Adherence"]

X_test_pca = pca_test.drop("Adherence", axis=1)
y_test_pca = pca_test["Adherence"]

print("X_train:", X_train_pca.shape)
print("X_test:", X_test_pca.shape)

X_train: (4000, 17)
X_test: (1000, 17)


In [4]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "SVM": SVC(
        probability=True,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        random_state=42,
        eval_metric="logloss"
    )
}

In [5]:
pca_predictions = {}
pca_probabilities = {}

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(X_train_pca, y_train_pca)

    pca_predictions[name] = model.predict(X_test_pca)
    pca_probabilities[name] = model.predict_proba(X_test_pca)[:, 1]

    print(f"{name} completed.\n")

Training Logistic Regression...
Logistic Regression completed.

Training Decision Tree...
Decision Tree completed.

Training Random Forest...
Random Forest completed.

Training SVM...


c:\Users\aojha\Downloads\medication-non-adherence-ml\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM completed.

Training XGBoost...
XGBoost completed.



In [6]:
pca_results = []

for name in models:

    y_pred = pca_predictions[name]
    y_prob = pca_probabilities[name]

    pca_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_pca, y_pred),
        "Precision": precision_score(y_test_pca, y_pred),
        "Recall": recall_score(y_test_pca, y_pred),
        "F1-Score": f1_score(y_test_pca, y_pred),
        "ROC-AUC": roc_auc_score(y_test_pca, y_prob)
    })

pca_results_df = pd.DataFrame(pca_results)

pca_results_df

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.586,0.560563,0.435449,0.490148,0.630076
1,Decision Tree,0.551,0.508969,0.496718,0.502769,0.546701
2,Random Forest,0.608,0.577566,0.529540,0.552511,0.625458
3,SVM,0.582,0.561129,0.391685,0.461340,0.585257
4,XGBoost,0.590,0.552573,0.540481,0.546460,0.609343
